[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cCoolBoox/IFCD107-alumnado/blob/main/sprints/S05/S05_01_numpy.ipynb)

# S05_01 · NumPy: arrays, vectorización y broadcasting
**Sprint 5 · Matemáticas y estadística en código · ⏱ 45 min**

## Qué vas a aprender
1. Qué es un **array** de NumPy y por qué es mucho más rápido que una lista de Python.
2. A **seleccionar** datos con índices, cortes y **máscaras booleanas**.
3. A **vectorizar**: operar con todos los datos a la vez, sin bucles.
4. Qué es el *broadcasting* y cómo se usan las semillas para que los números aleatorios sean reproducibles.

## Contexto TurisData
El alojamiento de referencia de *TurisData Canarias* nos da su serie diaria de 2023 a 2025 (`ocupacion_diaria.csv`):
porcentaje de ocupación, temperatura, si el día es festivo y precio medio. Vamos a explorarla con NumPy.
Los datos son sintéticos.

In [ ]:
# Preparación (ejecuta esta celda siempre la primera)
import os, urllib.request

BASE_URL = "https://raw.githubusercontent.com/cCoolBoox/IFCD107-alumnado/main/datos/"

def dato(nombre):
    """Devuelve la ruta local de un archivo de datos del curso (lo descarga si no está)."""
    for carpeta in ("datos", "../datos", "../../datos"):
        ruta = os.path.join(carpeta, nombre)
        if os.path.exists(ruta):
            return ruta
    os.makedirs("datos", exist_ok=True)
    ruta = os.path.join("datos", nombre)
    urllib.request.urlretrieve(BASE_URL + nombre, ruta)
    return ruta


In [ ]:
import time

import numpy as np
import matplotlib.pyplot as plt

SEMILLA = 42

## 1 · Arrays: la estructura básica
Un **array** es una tabla de números del mismo tipo. Tiene `shape` (forma), `dtype` (tipo) y `ndim` (dimensiones).
Cargamos las 4 columnas numéricas del CSV (la primera columna, la fecha, la dejamos fuera).

In [ ]:
tabla = np.genfromtxt(dato("ocupacion_diaria.csv"), delimiter=",", skip_header=1, usecols=(1, 2, 3, 4))
print("forma:", tabla.shape, "· tipo:", tabla.dtype, "· dimensiones:", tabla.ndim)
print(tabla[:3])                     # las tres primeras filas
# Columnas: 0 = ocupacion_pct, 1 = temperatura_c, 2 = festivo (0/1), 3 = precio_medio

**Indexación y cortes** (`array[filas, columnas]`): `:` significa "todo".
`tabla[0, 1]` es un valor; `tabla[:, 0]` es la columna 0 entera; `tabla[10:20, :2]` son las filas 10-19 y las 2 primeras columnas.

In [ ]:
print(tabla[0, 1], tabla[:3, 0], tabla[10:13, :2])

### Ejercicio 1 · Separar las columnas
Guarda en cuatro variables las columnas de `tabla`: `ocupacion`, `temperatura`, `festivo` y `precio`.

In [ ]:
# TODO: cada variable es una columna completa: tabla[:, k] con k = 0, 1, 2, 3
...
assert ocupacion.shape == (1096,), f"ocupacion debe tener 1096 valores (uno por día) y tiene forma {ocupacion.shape}"
assert temperatura.mean().round(2) == 21.10, "temperatura debe ser la columna 1 (temperatura media 21.10)"
assert festivo.sum() == 61, "festivo debe ser la columna 2 (hay 61 días festivos)"
assert precio.mean().round(2) == 87.52, "precio debe ser la columna 3 (precio medio 87.52)"
print("Correcto:", ocupacion.shape[0], "días")

## 2 · Máscaras booleanas: filtrar sin bucles
Una comparación sobre un array devuelve otro array de `True`/`False`. Usándolo como índice, nos quedamos
solo con los elementos `True`. Se combinan con `&` (y), `|` (o), `~` (no), siempre entre paréntesis.

In [ ]:
mascara_calor = temperatura > 25
print("Días de más de 25 °C:", mascara_calor.sum(), "· ocupación media esos días:", ocupacion[mascara_calor].mean().round(1))

### Ejercicio 2 · Festivos frente a laborables
Guarda en `media_festivos` la ocupación media de los días festivos y en `media_resto` la de los demás,
ambas redondeadas a 2 decimales (`.round(2)`).

In [ ]:
# TODO: máscara festivo == 1 para los festivos y festivo == 0 para el resto; luego .mean().round(2)
...
assert media_festivos == 83.88, f"La media en festivos es 83.88 y tienes {media_festivos}"
assert media_resto == 67.58, f"La media en días no festivos es 67.58 y tienes {media_resto}"
print(f"Festivos: {media_festivos} % · resto: {media_resto} % · diferencia: {media_festivos - media_resto:.2f} puntos")

## 3 · Vectorización: operar con todo a la vez
Sumar, multiplicar, comparar... se aplica **elemento a elemento** a todo el array. Es más corto y mucho más rápido
que un bucle de Python.

In [ ]:
grande = np.arange(1_000_000, dtype=float)

inicio = time.perf_counter()
suma_bucle = 0.0
for valor in grande.tolist():
    suma_bucle += valor * 2
t_bucle = time.perf_counter() - inicio

inicio = time.perf_counter()
suma_vector = (grande * 2).sum()
t_vector = time.perf_counter() - inicio

print(f"Bucle: {t_bucle*1000:.1f} ms · vectorizado: {t_vector*1000:.2f} ms · el resultado es el mismo: {suma_bucle == suma_vector}")

### Ejercicio 3 · Ingreso diario y estandarización
Suponiendo un alojamiento de **120 habitaciones**, el ingreso de cada día es
`ocupacion / 100 * 120 * precio`. Guarda ese array en `ingreso_diario`.
Después estandariza la ocupación (restar la media y dividir entre la desviación típica) en `ocupacion_z`.

In [ ]:
# TODO: operaciones directas entre arrays; para ocupacion_z usa ocupacion.mean() y ocupacion.std()
...
assert ingreso_diario.shape == (1096,), "ingreso_diario debe tener un valor por día"
assert np.isclose(ingreso_diario[0], 10405.524), f"El primer día: 0,89 x 120 x 97,43 = 10405.52 y tienes {ingreso_diario[0]}"
assert abs(ocupacion_z.mean()) < 1e-9 and abs(ocupacion_z.std() - 1) < 1e-9, "ocupacion_z debe tener media 0 y desviación típica 1"
print(f"Correcto. Ingreso medio diario: {ingreso_diario.mean():,.0f} EUR")

## 4 · Broadcasting: arrays de distinta forma
Cuando las formas no coinciden, NumPy **"estira"** el array pequeño para que encaje (si es posible).
Restar a una tabla `(1096, 4)` un vector de 4 medias (una por columna) funciona directamente.

In [ ]:
medias = tabla.mean(axis=0)          # axis=0: una media por columna -> forma (4,)
print(medias.round(2), medias.shape)
centrada = tabla - medias            # (1096, 4) - (4,) -> se resta a cada fila
print(centrada.mean(axis=0).round(6))

### Ejercicio 4 · Estandarizar todas las columnas a la vez
Guarda en `tabla_z` la tabla con **cada columna** estandarizada (media 0, desviación 1) usando broadcasting,
sin bucles. Pista: `tabla.mean(axis=0)` y `tabla.std(axis=0)`.

In [ ]:
# TODO: (tabla - medias por columna) / desviaciones por columna, con axis=0
...
assert tabla_z.shape == tabla.shape, f"tabla_z debe tener la misma forma que tabla {tabla.shape}"
assert np.allclose(tabla_z.mean(axis=0), 0, atol=1e-9), "Cada columna debe tener media 0 (¿usaste axis=0?)"
assert np.allclose(tabla_z.std(axis=0), 1), "Cada columna debe tener desviación típica 1"
print("Correcto")

## 5 · Números aleatorios reproducibles
`np.random.default_rng(semilla)` crea un generador. Con la **misma semilla** salen los **mismos** números:
clave para que tus resultados se puedan repetir (reproducibilidad).

In [ ]:
rng = np.random.default_rng(SEMILLA)
print(rng.normal(68, 10, 3).round(2))
rng = np.random.default_rng(SEMILLA)          # misma semilla, mismos números
print(rng.normal(68, 10, 3).round(2))

### Ejercicio 5 · Simulación reproducible
Con un generador de semilla `SEMILLA`, genera `simulada` = 365 ocupaciones aleatorias con distribución normal
de media 68,5 y desviación 9,6, y recórtalas al rango 0–100 con `np.clip`. Comprueba con un segundo generador
igual que el resultado se repite.

In [ ]:
# TODO: rng = np.random.default_rng(SEMILLA); rng.normal(media, desviación, n); np.clip(valores, 0, 100)
...
assert simulada.shape == (365,), "Debe haber 365 valores"
assert simulada.min() >= 0 and simulada.max() <= 100, "Los valores deben estar entre 0 y 100"
assert np.array_equal(simulada, simulada_b), "Con la misma semilla, los dos resultados deben ser idénticos"
assert abs(simulada.mean() - 68.5) < 2, "La media debería estar cerca de 68.5"
print("Correcto: media simulada", simulada.mean().round(2))

## 6 · Mini-reto integrador: media móvil de 7 días
La ocupación diaria oscila mucho. Calcula `media_movil` con la **media de cada ventana de 7 días consecutivos**
(una técnica que verás mucho en series temporales). Pista: `np.convolve(ocupacion, np.ones(7) / 7, mode="valid")`.
Después se dibuja junto a la serie original.

In [ ]:
# TODO: np.convolve con una ventana de 7 pesos iguales (1/7) y mode="valid"
...
assert media_movil.shape == (1090,), f"Con 1096 días y ventanas de 7 quedan 1090 valores y tienes {media_movil.shape}"
assert np.isclose(media_movil[0], ocupacion[:7].mean()), "El primer valor es la media de los 7 primeros días"

plt.figure(figsize=(10, 3.5))
plt.plot(ocupacion, color="lightgray", label="Ocupación diaria")
plt.plot(np.arange(6, 1096), media_movil, color="tab:blue", label="Media móvil 7 días")
plt.xlabel("Día (desde 1/1/2023)")
plt.ylabel("Ocupación (%)")
plt.title("Ocupación diaria y su media móvil de 7 días")
plt.legend()
plt.tight_layout()
plt.show()

## 7 · Para reflexionar
1. ¿Qué diferencia hay entre `ocupacion[festivo == 1]` y un bucle `for` con un `if`? ¿Cuándo preferirías cada uno?
2. ¿Por qué el ejercicio 4 necesita `axis=0`? ¿Qué pasaría con `axis=1`?
3. ¿Qué ventaja tiene fijar la semilla en una simulación que enseñas a un cliente?

## 8 · Qué has aprendido
- Un array tiene forma y tipo; se selecciona con índices, cortes y máscaras.
- Vectorizar = operar con todo el array sin bucles (más corto y más rápido).
- Broadcasting = operar arrays de formas distintas si son compatibles.
- `default_rng(semilla)` hace reproducibles las simulaciones.